In [32]:
OBJECTS = ["screwdriver", "hose", "valve", "valve_terminal", "box_w_screws", 
           "box_w_membrane", "soldering_station", "soldering_iron", "soldering_tin", 
           "soldering_board", "capacitor", "robot", "human", "RightHand", "eef_robot", 
           "robot_base"]

OBJECTS_fixed = ["screwdriver", "hose", "valve", "valve terminal", "box with screws", 
                 "box with membrane", "soldering station", "soldering iron", "soldering tin", 
                 "soldering board", "capacitor", "robot", "human", "RightHand", "eef robot", 
                 "robot base"]

RELATIONS = ['contact', 'above', 'below', 'left of', 'right of', 'behind of', 'in front of',
             'inside', 'surround', 'moving together', 'halting together', 'fixed moving together',
             'getting close', 'moving apart', 'stable']


ACTIONS = ['approach', 'grab_screwdriver', 'plug_screwdriver', 'grab_valve', 
           'screw_screwdriver', 'release_valve', 'grab_soldering_iron', 'plug_soldering_iron', 
           'retreat', 'join_screwdriver', 'grab_valve_terminal', 'plug_valve_terminal', 
           'place_screwdriver', 'grab_box_w_screws', 'place_box_w_screws', 'grab_hose', 
           'wait_for_robot', 'plug_hose', 'grab_box_w_membrane', 'grab_soldering_station', 
           'solder_hose', 'release_soldering_station', 'release_box_w_membrane']

ACTIONS_fixed = ['approach', 'grab screwdriver', 'plug screwdriver', 'grab valve', 
           'screw screwdriver', 'release valve', 'grab soldering iron', 'plug soldering iron', 
           'retreat', 'join screwdriver', 'grab valve terminal', 'plug valve terminal', 
           'place screwdriver', 'grab box with screws', 'place box with screws', 'grab hose', 
           'wait for robot', 'plug hose', 'grab box with membrane', 'grab soldering station', 
           'solder hose', 'release soldering station', 'release box with membrane']


bim_OBJECTS = ['bowl', 'knife', 'screwdriver', 'cuttingboard', 'whisk', 'hammer', 'bottle', 'cup',
           'banana', 'cereals', 'sponge', 'woodenwedge', 'saw', 'harddrive', 'LeftHand',
           'RightHand']


bim_ACTIONS = ['idle', 'approach', 'retreat', 'lift', 'place', 'hold', 'pour', 'cut', 'hammer', 'saw',
           'stir', 'screw', 'drink', 'wipe']


# map all names to their expanded names
item_mapping = {}

item_mapping.update({name: name for name in bim_OBJECTS})
item_mapping.update({name: name for name in bim_ACTIONS})
item_mapping.update({name: name for name in RELATIONS})

item_mapping.update({name: expanded for name, expanded in zip(OBJECTS, OBJECTS_fixed)})
item_mapping.update({name: expanded for name, expanded in zip(ACTIONS, ACTIONS_fixed)})

print(len(item_mapping))
item_mapping


78


{'bowl': 'bowl',
 'knife': 'knife',
 'screwdriver': 'screwdriver',
 'cuttingboard': 'cuttingboard',
 'whisk': 'whisk',
 'hammer': 'hammer',
 'bottle': 'bottle',
 'cup': 'cup',
 'banana': 'banana',
 'cereals': 'cereals',
 'sponge': 'sponge',
 'woodenwedge': 'woodenwedge',
 'saw': 'saw',
 'harddrive': 'harddrive',
 'LeftHand': 'LeftHand',
 'RightHand': 'RightHand',
 'idle': 'idle',
 'approach': 'approach',
 'retreat': 'retreat',
 'lift': 'lift',
 'place': 'place',
 'hold': 'hold',
 'pour': 'pour',
 'cut': 'cut',
 'stir': 'stir',
 'screw': 'screw',
 'drink': 'drink',
 'wipe': 'wipe',
 'contact': 'contact',
 'above': 'above',
 'below': 'below',
 'left of': 'left of',
 'right of': 'right of',
 'behind of': 'behind of',
 'in front of': 'in front of',
 'inside': 'inside',
 'surround': 'surround',
 'moving together': 'moving together',
 'halting together': 'halting together',
 'fixed moving together': 'fixed moving together',
 'getting close': 'getting close',
 'moving apart': 'moving apart',


In [39]:
from sentence_transformers import SentenceTransformer

model_name = "all-MiniLM-L6-v2"
model = SentenceTransformer(model_name)
ref_embeddings = model.encode(list(item_mapping.values()))

In [43]:
ret = model.encode(["left of", "right of"])

cosine_similarity(ret[0].reshape(1, -1), ret[1].reshape(1, -1))

array([[0.750018]], dtype=float32)

In [37]:
import numpy as np
from sklearn.decomposition import PCA
from sklearn.metrics.pairwise import cosine_similarity

pca = PCA(n_components=16)
pca.fit(ref_embeddings)
ref_embeddings_pca = pca.transform(ref_embeddings)

recovered_embeddings = pca.inverse_transform(ref_embeddings_pca)

error = np.mean((ref_embeddings - recovered_embeddings)**2)

exp_var = pca.explained_variance_.sum()


# Compute pairwise cosine similarity
original_similarity = cosine_similarity(ref_embeddings)
reduced_similarity = cosine_similarity(ref_embeddings_pca)

# Compute correlation between original and reduced similarities
correlation = np.corrcoef(original_similarity.flatten(), reduced_similarity.flatten())[0, 1]

print(f"Error: {error:.2f}")
print(f"Explained variance: {exp_var:.2f}")
print(f"Similarity preservation correlation: {correlation:.2f}")


Error: 0.00
Explained variance: 0.51
Similarity preservation correlation: 0.90
